# Vérification du téléchargement du cadastre

Ce notebook vérifie que `download.cadastre.download_cadastre` récupère les parcelles cadastrales (Parcellaire Express IGN, WFS `CADASTRALPARCELS.PARCELLAIRE_EXPRESS:parcelle`), filtrées sur le code INSEE exact de l'entité, et affiche le résultat sur une carte.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from core.config import load_entities
from download.limites_administratives import fetch_commune_boundary, resolve_bbox
from download.cadastre import download_cadastre

## 1. Entité testée (issue du CSV)

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
entity

## 2. Téléchargement des parcelles cadastrales de la commune

In [ ]:
bbox = resolve_bbox(entity)
written = download_cadastre(entity.code_insee, bbox)
written

## 3. Relecture du GeoPackage écrit

In [ ]:
import geopandas as gpd

parcelles = gpd.read_file(written)
print("Nombre de parcelles :", len(parcelles))
parcelles[["numero", "section", "contenance", "code_insee"]].head(10)

## 4. Affichage sur une carte `leafmap`

Reprojection en EPSG:4326 uniquement pour l'affichage. Backend `folium` (HTML statique), plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import leafmap.foliumap as leafmap

boundary_wgs84 = fetch_commune_boundary(entity.code_insee).to_crs(epsg=4326)
parcelles_wgs84 = parcelles.to_crs(epsg=4326)

m = leafmap.Map()
m.add_gdf(
    parcelles_wgs84,
    layer_name="Parcelles cadastrales",
    style={"color": "black", "weight": 1, "fillOpacity": 0.05},
    info_mode="on_click",
)
m.add_gdf(
    boundary_wgs84,
    layer_name="Limite communale",
    style={"color": "red", "fillOpacity": 0},
)
m.zoom_to_gdf(boundary_wgs84)
m